# LaptopCare AI — Model Experiment

This notebook documents the text-classification experiment behind the diagnosis engine:

`Raw Complaint -> Text Cleaning -> TF-IDF -> Classifier -> Probability -> Prediction`

It compares three candidate models (Logistic Regression, Linear SVM, Naive Bayes) using
5-fold stratified cross-validation on an 80/20 train/test split, then reports
accuracy, precision, recall, F1-score, and a confusion matrix for the best one.

This mirrors exactly what `app/ml/train.py` does programmatically — this notebook is for
exploration/portfolio purposes.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from app.ml.preprocess import clean_text

In [ ]:
df = pd.read_csv('../dataset/laptop_complaints.csv')
df['text'] = df['text'].apply(clean_text)
df['label'].value_counts()

In [ ]:
X, y = df['text'], df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
len(X_train), len(X_test)

## Model comparison (5-fold CV, F1-macro)

In [ ]:
candidates = {
    'logistic_regression': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
        ('clf', LogisticRegression(max_iter=1000)),
    ]),
    'linear_svm': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
        ('clf', LinearSVC()),
    ]),
    'naive_bayes': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
        ('clf', MultinomialNB()),
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, pipe in candidates.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1_macro')
    print(f"{name:20s} F1-macro = {scores.mean():.4f} (+/- {scores.std():.4f})")

## Final evaluation on the held-out test set (best candidate)

In [ ]:
best_pipeline = candidates['logistic_regression']  # swap based on the comparison above
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

## Try a sample complaint

In [ ]:
sample = clean_text("Laptop saya panas sekali dan mati sendiri setelah dipakai 30 menit")
pred = best_pipeline.predict([sample])[0]
proba = dict(zip(best_pipeline.classes_, best_pipeline.predict_proba([sample])[0]))
print('Prediction:', pred)
print('Top 3:', sorted(proba.items(), key=lambda kv: kv[1], reverse=True)[:3])